# Trabalho Prático 1 - Métodos de Busca (UCS e A*)
**Departamento de Informática e Estatística - UFSC**  
**Curso:** Sistemas de Informação  
**Disciplina:** Sistemas Inteligentes  
**Professora:** Nathalia da Cruz Alves  

---

### Identificação da Equipe (A1.1)
- **Estudante 1:** Luiz Eduardo — *Contribuição: Modelagem do espaço de estados, implementação da classe Maze e gerador de instâncias.*
- **Estudante 2:** [Nome do Estudante] — *Contribuição: Implementação do algoritmo de busca genérico (UCS e A*), formulação das heurísticas e coleta das métricas.*
- **Estudante 3 (opcional):** [Nome do Estudante] — *Contribuição: Estruturação dos cenários fácil e difícil, análise comparativa e roteiro do vídeo.*

---

## 1. Descrição do Problema e Formulação no Espaço de Estados (A1.1)

O problema escolhido é a **Navegação em Labirinto em uma Matriz 2D** (Tema 2.a dos requisitos):
- **Espaço de Estados:** O conjunto de todas as coordenadas discretas `(linha, coluna)` no grid bidimensional de dimensões $H 	imes W$. Cada célula representa uma posição no labirinto.
- **Estado Inicial:** A coordenada de partida `start = (r_start, c_start)`.
- **Estado / Condição Objetivo:** A coordenada de destino `goal = (r_goal, c_goal)`.
- **Ações / Movimentos Possíveis:** O agente pode se mover ortogonalmente em até 4 direções:
  - Cima: `(r - 1, c)`
  - Baixo: `(r + 1, c)`
  - Esquerda: `(r, c - 1)`
  - Direita: `(r, c + 1)`
  Um movimento é válido se e somente se a posição de destino estiver dentro dos limites da matriz e não pertencer ao conjunto de paredes (`walls`).
- **Custo das Ações:** Custo unitário uniforme ($c = 1.0$) para cada deslocamento entre células adjacentes livres.


In [1]:
from __future__ import annotations

import heapq
import math
import time
from random import Random
from typing import Iterable, Optional, Sequence

# Tipo para coordenadas discretas (linha, coluna)
Position = tuple[int, int]


In [ ]:

class Maze:
    """Representação de um labirinto em grade retangular.
    
    - As dimensões (altura e largura) são informadas no construtor.
    - As paredes ('walls') são informadas exclusivamente como uma lista/coleção 
      de coordenadas (linha, coluna).
    - 'start' e 'goal' são obrigatórios.
    """

    WALL = "#"
    OPEN = " "
    START = "S"
    GOAL = "G"
    PATH = "·"

    def __init__(
        self,
        height: int,
        width: int,
        walls: Iterable[Position],
        start: Position,
        goal: Position,
    ) -> None:
        if isinstance(height, bool) or isinstance(width, bool):
            raise TypeError("height e width devem ser inteiros")
        if not isinstance(height, int) or not isinstance(width, int):
            raise TypeError("height e width devem ser inteiros")
        if height < 2 or width < 2:
            raise ValueError("o labirinto precisa ter pelo menos 2 linhas e 2 colunas")
        if walls is None:
            raise ValueError("walls não pode ser None")
        if start is None:
            raise ValueError("start não pode ser None")
        if goal is None:
            raise ValueError("goal não pode ser None")

        self.height = self.rows = height
        self.width = self.columns = width
        self.walls: set[Position] = set(walls)

        for wall_pos in self.walls:
            self._validate_position(wall_pos, "wall")

        self.start = start
        self.goal = goal
        self._validate_position(self.start, "start")
        self._validate_position(self.goal, "goal")

        if self.is_wall(self.start) or self.is_wall(self.goal):
            raise ValueError("start e goal devem ser posições livres (não paredes)")

    @property
    def size(self) -> tuple[int, int]:
        """Retorna as dimensões (altura, largura)."""
        return self.rows, self.columns

    def _validate_position(self, position: Position, label: str = "position") -> None:
        if (
            not isinstance(position, tuple)
            or len(position) != 2
            or not all(isinstance(v, int) for v in position)
        ):
            raise TypeError(f"{label} deve ser uma tupla (linha, coluna)")
        r, c = position
        if not (0 <= r < self.rows and 0 <= c < self.columns):
            raise ValueError(f"{label} {position} está fora dos limites do labirinto")

    def is_wall(self, position: Position) -> bool:
        """Verifica se a coordenada é uma parede."""
        self._validate_position(position)
        return position in self.walls

    def is_walkable(self, position: Position) -> bool:
        """Verifica se a célula está nos limites e livre para passagem."""
        r, c = position
        return (
            0 <= r < self.rows
            and 0 <= c < self.columns
            and position not in self.walls
        )

    def neighbors(self, position: Position) -> Iterable[Position]:
        """Gera os vizinhos válidos nos 4 sentidos (Cima, Direita, Baixo, Esquerda)."""
        r, c = position
        for next_pos in (
            (r - 1, c),
            (r, c + 1),
            (r + 1, c),
            (r, c - 1),
        ):
            if self.is_walkable(next_pos):
                yield next_pos

    def get_matrix(
        self, path: Optional[Sequence[Position]] = None
    ) -> list[list[str]]:
        """Retorna a matriz 2D de células.
        
        Se um caminho for informado, as células do caminho são preenchidas com
        os índices sequenciais dos passos, iniciando em '0' no start.
        """
        path_map: dict[Position, str] = {}
        if path is not None:
            path_list = list(path)
            if path_list and self.start not in path_list:
                path_list = [self.start] + path_list
            for step, pos in enumerate(path_list):
                path_map[pos] = str(step)

        matrix: list[list[str]] = []
        for r in range(self.rows):
            line: list[str] = []
            for c in range(self.columns):
                pos = (r, c)
                if pos in path_map:
                    line.append(path_map[pos])
                elif pos == self.start:
                    line.append(self.START)
                elif pos == self.goal:
                    line.append(self.GOAL)
                elif pos in self.walls:
                    line.append(self.WALL)
                else:
                    line.append(self.OPEN)
            matrix.append(line)
        return matrix

    def render(self, path: Optional[Sequence[Position]] = None) -> str:
        """Retorna a representação textual formatada e alinhada do labirinto."""
        matrix = self.get_matrix(path)
        max_len = max(len(cell) for row in matrix for cell in row)
        use_spaces = path is not None or max_len > 1
        if use_spaces:
            text = "\n".join(
                " ".join(f"{cell:>{max_len}}" for cell in row) for row in matrix
            )
        else:
            text = "\n".join("".join(row) for row in matrix)
        return text

Classe Maze compilada com sucesso!


In [ ]:
class MazeGenerator:
    """Fábrica de labirintos conectados usando recursive backtracking."""

    def __init__(
        self,
        height: int,
        width: int,
        *,
        seed: Optional[int] = None,
        loop_probability: float = 0.0,
    ) -> None:
        self._validate_dimension(height, "height")
        self._validate_dimension(width, "width")
        if height < 2 or width < 2:
            raise ValueError("o labirinto precisa ter pelo menos 2 linhas e 2 colunas")
        if not 0.0 <= loop_probability <= 1.0:
            raise ValueError("loop_probability deve estar entre 0 e 1")

        self.height = self.rows = height
        self.width = self.columns = width
        self.loop_probability = loop_probability
        self.random = Random(seed)

    @staticmethod
    def _validate_dimension(value: int, name: str) -> None:
        """Valida uma dimensão do labirinto."""
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} deve ser um inteiro")

    def generate(self) -> "Maze":
        """Gera e retorna uma nova instância de Maze."""
        lattice = self._lattice_cells()
        if len(lattice) < 2:
            return self._create_open_maze()

        grid = self._create_wall_grid()
        self._carve_passages(grid, lattice)
        self._add_loops(grid)

        start = lattice[0]
        goal = self._farthest_cell(grid, start)
        walls = self._wall_coordinates(grid)

        return Maze(
            height=self.height,
            width=self.width,
            walls=walls,
            start=start,
            goal=goal,
        )

    def _lattice_cells(self) -> list[Position]:
        """Retorna as células internas usadas como pontos do labirinto."""
        return [
            (row, column)
            for row in range(1, self.height - 1, 2)
            for column in range(1, self.width - 1, 2)
        ]

    def _create_wall_grid(self) -> list[list[bool]]:
        """Cria uma grade em que True representa parede e False representa passagem."""
        return [
            [True for _ in range(self.width)]
            for _ in range(self.height)
        ]

    def _carve_passages(
        self,
        grid: list[list[bool]],
        lattice: list[Position],
    ) -> None:
        """Abre corredores conectando todas as células da lattice."""
        start = lattice[0]
        visited = {start}
        stack = [start]
        grid[start[0]][start[1]] = False

        while stack:
            current = stack[-1]
            candidates = [
                cell
                for cell in self._lattice_neighbors(current)
                if cell not in visited
            ]

            if not candidates:
                stack.pop()
                continue

            next_cell = self.random.choice(candidates)
            between = (
                (current[0] + next_cell[0]) // 2,
                (current[1] + next_cell[1]) // 2,
            )

            grid[between[0]][between[1]] = False
            grid[next_cell[0]][next_cell[1]] = False
            visited.add(next_cell)
            stack.append(next_cell)

    def _lattice_neighbors(self, cell: Position) -> list[Position]:
        """Retorna os vizinhos da lattice separados por uma célula."""
        row, column = cell
        neighbors = []
        for next_cell in (
            (row - 2, column),
            (row, column + 2),
            (row + 2, column),
            (row, column - 2),
        ):
            next_row, next_column = next_cell
            if (
                1 <= next_row < self.height - 1
                and 1 <= next_column < self.width - 1
            ):
                neighbors.append(next_cell)
        return neighbors

    def _add_loops(self, grid: list[list[bool]]) -> None:
        """Remove algumas paredes internas para criar caminhos alternativos."""
        for row in range(1, self.height - 1):
            for column in range(1, self.width - 1):
                if not grid[row][column]:
                    continue
                if self.random.random() > self.loop_probability:
                    continue

                open_neighbors = sum(
                    not grid[next_row][next_column]
                    for next_row, next_column in (
                        (row - 1, column),
                        (row + 1, column),
                        (row, column - 1),
                        (row, column + 1),
                    )
                    if 0 <= next_row < self.height
                    and 0 <= next_column < self.width
                )
                if open_neighbors >= 2:
                    grid[row][column] = False

    def _farthest_cell(self, grid: list[list[bool]], start: Position) -> Position:
        """Encontra a célula aberta mais distante de start por busca em largura."""
        distances = {start: 0}
        queue = [start]

        for current in queue:
            row, column = current
            for neighbor in (
                (row - 1, column),
                (row, column + 1),
                (row + 1, column),
                (row, column - 1),
            ):
                next_row, next_column = neighbor
                if not (
                    0 <= next_row < self.height
                    and 0 <= next_column < self.width
                ):
                    continue
                if grid[next_row][next_column] or neighbor in distances:
                    continue

                distances[neighbor] = distances[current] + 1
                queue.append(neighbor)

        return max(distances.items(), key=lambda item: item[1])[0]

    def _wall_coordinates(self, grid: list[list[bool]]) -> set[Position]:
        """Converte a grade booleana no conjunto de paredes do Maze."""
        return {
            (row, column)
            for row in range(self.height)
            for column in range(self.width)
            if grid[row][column]
        }

    def _create_open_maze(self) -> "Maze":
        """Cria um labirinto aberto para dimensões sem lattice suficiente."""
        return Maze(
            height=self.height,
            width=self.width,
            walls=[],
            start=(0, 0),
            goal=(self.height - 1, self.width - 1),
        )


print("Classe MazeGenerator compilada com sucesso!")


Classe MazeGenerator compilada com sucesso!


## Funcionamento teórico do `MazeGenerator`

O `MazeGenerator` funciona como uma **factory de labirintos**: ele recebe as dimensões do ambiente e devolve um objeto `Maze` completo, contendo `walls`, `start`, `goal`, `height` e `width`.

### Representação do estado

Durante a geração, o labirinto é representado por uma grade booleana:

- `True` representa uma parede;
- `False` representa uma passagem.

A grade começa completamente preenchida por paredes. Em seguida, o algoritmo abre apenas os corredores necessários.

As células principais do labirinto são obtidas por `_lattice_cells()`. Elas ficam em coordenadas internas separadas por duas posições, como `(1, 1)`, `(1, 3)` e `(3, 1)`. A posição entre duas células da lattice representa a parede que será removida para conectá-las.

### Algoritmo utilizado: Recursive Backtracking

A geração usa o algoritmo **Recursive Backtracking**, uma variação de busca em profundidade (`DFS`) aleatória. Nesta implementação, a recursão foi substituída por uma lista chamada `stack`, que guarda o caminho atual.

O algoritmo mantém duas estruturas principais:

- `visited`: conjunto das células da lattice que já foram visitadas;
- `stack`: caminho de células que ainda pode ser explorado.

A cada iteração:

1. A célula no topo da pilha é considerada a célula atual.
2. O gerador procura vizinhos da lattice que ainda não foram visitados.
3. Se existir um vizinho, um deles é escolhido aleatoriamente.
4. A parede entre a célula atual e o vizinho é removida.
5. O vizinho é marcado como visitado e colocado na pilha.
6. Se não houver vizinhos disponíveis, a célula é removida da pilha. Esse é o retrocesso (*backtracking*).

Como cada nova célula só é alcançada a partir de uma célula já conectada, todas as células abertas pertencem ao mesmo componente. Por isso, o labirinto gerado possui um caminho entre o início e todas as demais áreas abertas.

### Escolha do início e do objetivo

O início é a primeira célula da lattice, normalmente `(1, 1)`. Depois que os corredores são construídos, `_farthest_cell()` executa uma busca em largura (`BFS`) a partir do início e escolhe como objetivo a célula aberta mais distante.

Essa escolha cria um problema de busca mais interessante do que escolher um objetivo aleatório, pois o caminho tende a atravessar uma parte maior do labirinto.

Quando `loop_probability` é maior que zero, `_add_loops()` remove algumas paredes internas que possuem pelo menos duas passagens vizinhas. Isso cria caminhos alternativos e deixa o labirinto menos parecido com uma árvore.

### Fluxo de execução

```plantuml
@startuml
start
:Validar altura, largura e loop_probability;
:Construir células da lattice;

if (Há pelo menos duas células?) then (não)
  :Criar Maze aberto;
  stop
else (sim)
  :Criar grade preenchida com paredes;
  :Escolher a primeira célula;
  :Marcar início como passagem;
  :Adicionar início à stack e ao visited;
endif

while (A stack não está vazia?) is (sim)
  :Obter célula no topo da stack;
  :Procurar vizinhos da lattice não visitados;

  if (Existe vizinho disponível?) then (sim)
    :Escolher um vizinho aleatoriamente;
    :Remover a parede entre as duas células;
    :Abrir e visitar o vizinho;
    :Adicionar o vizinho à stack;
  else (não)
    :Remover a célula da stack;
  endif
endwhile (não)

if (loop_probability > 0?) then (sim)
  :Remover algumas paredes internas;
endif

:Encontrar a passagem mais distante com BFS;
:Converter a grade em conjunto de walls;
:Retornar Maze(start, goal, walls, height, width);
stop
@enduml
```

### Complexidade

Para uma grade com `H` linhas e `W` colunas:

- **Tempo:** `O(H × W)`, pois a grade é percorrida para gerar os corredores, procurar o objetivo e coletar as paredes;
- **Memória:** `O(H × W)`, devido à grade, ao conjunto `visited`, à fila da BFS e à `stack`;
- **Resultado:** uma instância de `Maze` pronta para ser usada pelos algoritmos UCS e A*.


## 2. Algoritmos e Heurísticas (A1.2 e A2.2 / A2.3)

O algoritmo **A\*** avalia cada nó pela função:
$$f(n) = g(n) + h(n)$$
onde:
- $g(n)$ é o custo real acumulado do nó inicial até o nó $n$.
- $h(n)$ é a estimativa heurística do custo restante de $n$ até o objetivo.

### As 4 Variações Consideradas

Cada heurística abaixo é documentada em uma célula Markdown própria e implementada imediatamente na célula de código seguinte.


### 1. UCS — Busca de Custo Uniforme

O UCS não utiliza informação heurística para estimar a distância até o objetivo. Por isso:

$$h_{UCS}(n) = 0$$

A função de prioridade torna-se:

$$f(n) = g(n) + h(n) = g(n)$$

Assim, o algoritmo sempre escolhe o estado com menor custo acumulado desde o início. Como todos os movimentos do `Maze` possuem custo unitário, o UCS expande os estados em ordem crescente do número de passos.

Essa heurística é admissível porque nunca superestima o custo restante: ela sempre retorna zero.


In [ ]:
# Heurística do UCS (Uniform Cost Search)
#
# O UCS não estima o custo restante até o objetivo. Por isso, h(n) = 0
# para qualquer estado n. A escolha dos estados será feita apenas pelo
# custo acumulado g(n), controlado pela futura fila de prioridade.

def h_ucs(position: Position, goal: Position) -> float:
    """Retorna zero para qualquer estado, como definido pelo UCS."""
    return 0.0


### 2. A* com Heurística Euclidiana Admissível

A distância Euclidiana mede a distância em linha reta entre a posição atual e o objetivo:

$$h_{eucl}(n) = \sqrt{(r_n - r_{goal})^2 + (c_n - c_{goal})^2}$$

Ela é admissível porque a distância real em um `Maze` com movimentos ortogonais nunca pode ser menor que a distância em linha reta:

$$0 \le h_{eucl}(n) \le h^*(n)$$

Portanto, o A* mantém a garantia de encontrar um caminho ótimo quando utiliza essa heurística.


In [ ]:
# Heurística Euclidiana (admissível)
#
# Representa a distância em linha reta entre a posição atual e o objetivo.

def h_admissible_euclidean(position: Position, goal: Position) -> float:
    """Retorna a distância Euclidiana entre position e goal."""
    row_distance = position[0] - goal[0]
    column_distance = position[1] - goal[1]
    return math.sqrt(row_distance ** 2 + column_distance ** 2)


### 3. A* com Heurística Manhattan

A distância Manhattan soma as diferenças absolutas entre as linhas e as colunas:

$$h_{manh}(n) = |r_n - r_{goal}| + |c_n - c_{goal}|$$

Como o agente se movimenta apenas na vertical e na horizontal, essa heurística representa o menor número de movimentos possível quando não existem paredes.

Ela é admissível e consistente. Além disso, domina a heurística Euclidiana:

$$h_{eucl}(n) \le h_{manh}(n) \le h^*(n)$$

Por ser mais informativa sem superestimar o custo, tende a expandir menos estados que a heurística Euclidiana.


In [ ]:
# Heurística Manhattan (admissível e consistente)
#
# Em um Maze com movimentos apenas ortogonais, soma as diferenças
# de linha e coluna até o objetivo.

def h_consistent_manhattan(position: Position, goal: Position) -> float:
    """Retorna a distância Manhattan entre position e goal."""
    row_distance = abs(position[0] - goal[0])
    column_distance = abs(position[1] - goal[1])
    return float(row_distance + column_distance)


### 4. A* com Heurística Manhattan Ponderada

A heurística ponderada multiplica a distância Manhattan por um peso `w` maior que 1:

$$h_{inadm}(n) = w \times h_{manh}(n), \quad w > 1$$

Nesta implementação, o peso padrão é `w = 4.0`.

Como o valor pode ser maior que o custo real restante, essa heurística não é admissível. Por exemplo, se a distância Manhattan for `2`, a estimativa ponderada será `8` quando `w = 4`.

Ela pode direcionar a busca mais agressivamente ao objetivo, mas perde a garantia teórica de encontrar sempre o caminho ótimo.


In [ ]:
# Heurística Manhattan ponderada (não admissível)
#
# O peso maior que 1 aumenta artificialmente a estimativa e pode
# fazer o A* priorizar caminhos aparentemente mais próximos do objetivo.

def h_inadmissible_weighted(
    position: Position,
    goal: Position,
    w: float = 4.0,
) -> float:
    """Retorna a distância Manhattan multiplicada por um peso maior que 1."""
    if w <= 1.0:
        raise ValueError("w deve ser maior que 1 para ser não admissível")
    return w * h_consistent_manhattan(position, goal)


### Gerenciamento da Fronteira (A2.2)

A futura implementação da busca deverá utilizar uma fila de prioridade (`heapq`) ordenada pelo menor valor de $f(n)$. Em caso de empate, podem ser usados o valor de $h(n)$ e um contador sequencial.

Antes de inserir um sucessor na fronteira, deve-se verificar o menor custo `g(n)` conhecido para aquela posição:

- se o estado ainda não foi visitado, ele pode ser inserido;
- se já foi visitado, ele só deve ser reinserido quando o novo caminho tiver custo acumulado menor.
